# Using the BaseDecorator Class in baseobjects

## Introduction

The `BaseDecorator` class provides a foundation for creating Python decorators with extended functionality. It extends the `BaseFunction` class to create decorator-like callable objects that can handle both regular functions and coroutine functions. One of its key features is supporting both simple decorators and decorators that accept arguments.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `BaseDecorator` class
- Creating simple decorators using `BaseDecorator`
- Creating decorators that accept arguments
- Handling both regular functions and coroutine functions
- Subclassing `BaseDecorator` to create custom decorators

**Prerequisites:**
- Basic understanding of Python decorators
- Familiarity with Python's callable objects
- Understanding of function annotations and type hints
- Basic knowledge of coroutines and async/await syntax (for advanced examples)

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.functions import BaseDecorator

## Core Functionality

The `BaseDecorator` class extends Python's decorator capabilities by providing a structured way to create both simple decorators and decorators that accept arguments. Let's explore the core functionality and understand how it works.

### Basic Concept

In Python, decorators are a powerful way to modify or enhance the behavior of functions or methods. The `BaseDecorator` class provides a foundation for creating decorators with extended functionality, handling both regular functions and coroutine functions.

Key features of `BaseDecorator`:
1. It can be used directly as a decorator or subclassed to create custom decorators
2. It supports both standard decorator usage (`@decorator`) and parameterized decorator usage (`@decorator(arg1=value1)`)
3. It handles both regular functions and coroutine functions
4. It includes methods for proper pickling and unpickling

Let's start with a simple example of using `BaseDecorator` directly:

In [2]:
# Creating a simple decorator using BaseDecorator
class SimpleDecorator(BaseDecorator):
    def __call__(self, *args, **kwargs):
        print(f"Before calling {self.__wrapped__.__name__}")
        result = self.__wrapped__(*args, **kwargs)
        print(f"After calling {self.__wrapped__.__name__}")
        return result


# Using the decorator
@SimpleDecorator
def greet(name) -> str:
    print(f"Hello, {name}!")
    return f"Greeted {name}"


# Call the decorated function
result = greet("World")
print(f"Result: {result}")

Before calling greet
Hello, World!
After calling greet
Result: Greeted World


In the example above, we created a simple decorator that prints messages before and after calling the decorated function. The `BaseDecorator` class handles the wrapping of the function, and we only need to implement the `__call__` method to define the behavior of our decorator.

### Creating Decorators with Arguments

One of the powerful features of `BaseDecorator` is its ability to handle decorators that accept arguments. Let's create a decorator that repeats the function call a specified number of times:

In [3]:
# Creating a decorator with arguments
class RepeatDecorator(BaseDecorator):
    def __init__(self, func=None, times=2) -> None:
        super().__init__(func)
        self.times = times

    def __call__(self, *args, **kwargs):
        results = []
        for _ in range(self.times):
            results.append(self.__wrapped__(*args, **kwargs))
        return results


# Using the decorator without arguments (default times=2)
@RepeatDecorator
def say_hello(name) -> str:
    return f"Hello, {name}!"


# Using the decorator with arguments
@RepeatDecorator(times=3)
def say_goodbye(name) -> str:
    return f"Goodbye, {name}!"


# Call the decorated functions
hello_results = say_hello("Alice")
goodbye_results = say_goodbye("Bob")

print("Hello results:", hello_results)
print("Goodbye results:", goodbye_results)

Hello results: ['Hello, Alice!', 'Hello, Alice!']
Goodbye results: ['Goodbye, Bob!', 'Goodbye, Bob!', 'Goodbye, Bob!']


In this example, we created a decorator that repeats the function call a specified number of times. The `BaseDecorator` class handles the dual-mode behavior of decorators:
1. When used without arguments (`@RepeatDecorator`), it uses the default value for `times`.
2. When used with arguments (`@RepeatDecorator(times=3)`), it creates a decorator with the specified value for `times`.

### How BaseDecorator Works

The `BaseDecorator` class implements the dual-mode behavior of decorators through its `__new__` method. When called with a function, it creates and returns a decorator instance that wraps that function. When called without a function, it returns a partial function that will create a decorator instance when later called with a function.

This enables both standard decorator usage (`@decorator`) and parameterized decorator usage (`@decorator(arg1=value1)`).

## Module Interaction

The `BaseDecorator` class extends `BaseFunction` from the `baseobjects.bases` module, inheriting its functionality for creating callable objects. It also uses the `AnyCallable` type from `baseobjects.typing` for type hints.

Let's see how `BaseDecorator` interacts with other components of the baseobjects package:

In [4]:
from baseobjects.bases import BaseFunction
from baseobjects.typing import AnyCallable
from baseobjects.functions import DynamicFunction


# Creating a decorator that uses DynamicFunction
class DynamicDecorator(BaseDecorator):
    def __init__(self, func=None, prefix="", suffix="") -> None:
        super().__init__(func)
        self.prefix = prefix
        self.suffix = suffix

    def __call__(self, *args, **kwargs):
        # Create a dynamic function that wraps the original function
        def wrapper(*inner_args, **inner_kwargs):
            result = self.__wrapped__(*inner_args, **inner_kwargs)
            if isinstance(result, str):
                return f"{self.prefix}{result}{self.suffix}"
            return result

        dynamic_func = DynamicFunction(wrapper)
        return dynamic_func(*args, **kwargs)


# Using the decorator
@DynamicDecorator(prefix="[START] ", suffix=" [END]")
def process_text(text):
    return text.upper()


# Call the decorated function
result = process_text("hello world")
print(result)

[START] HELLO WORLD [END]


In this example, we created a decorator that uses `DynamicFunction` from the `baseobjects.functions` module to create a dynamic function that wraps the original function. This demonstrates how `BaseDecorator` can interact with other components of the baseobjects package.

## Advanced Features

### Handling Coroutine Functions

The `BaseDecorator` class can handle both regular functions and coroutine functions. Let's create a decorator that works with async functions:

In [5]:
import asyncio
import time

# Import nest_asyncio to allow coroutines in Jupyter Notebooks
import nest_asyncio

nest_asyncio.apply()


# Creating a decorator for timing async functions
class AsyncTimerDecorator(BaseDecorator):
    async def __call__(self, *args, **kwargs):
        start_time = time.time()
        result = await self.__wrapped__(*args, **kwargs)
        end_time = time.time()
        print(f"{self.__wrapped__.__name__} took {end_time - start_time:.4f} seconds to execute")
        return result


# Using the decorator with an async function
@AsyncTimerDecorator
async def fetch_data(delay):
    print(f"Fetching data with delay {delay} seconds...")
    await asyncio.sleep(delay)  # Simulate network delay
    return {"data": "Some fetched data", "delay": delay}


# Run the async function
async def main() -> None:
    result = await fetch_data(1.5)
    print(f"Result: {result}")


# Execute the async function
asyncio.run(main())

Fetching data with delay 1.5 seconds...
fetch_data took 1.5074 seconds to execute
Result: {'data': 'Some fetched data', 'delay': 1.5}


In this example, we created a decorator that measures the execution time of async functions. The `BaseDecorator` class handles the wrapping of the coroutine function, and we implement an async `__call__` method to define the behavior of our decorator.

## Examples

### Example 1: Creating a Logging Decorator

Let's create a decorator that logs function calls with their arguments and return values:

In [6]:
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger("function_logger")


# Creating a logging decorator
class LoggingDecorator(BaseDecorator):
    def __init__(self, func=None, log_level=logging.INFO) -> None:
        super().__init__(func)
        self.log_level = log_level

    def __call__(self, *args, **kwargs):
        # Log function call with arguments
        args_repr = [repr(a) for a in args]
        kwargs_repr = [f"{k}={v!r}" for k, v in kwargs.items()]
        signature = ", ".join(args_repr + kwargs_repr)
        logger.log(self.log_level, f"Calling {self.__wrapped__.__name__}({signature})")

        # Call the function
        try:
            result = self.__wrapped__(*args, **kwargs)
            # Log the return value
            logger.log(self.log_level, f"{self.__wrapped__.__name__} returned {result!r}")
            return result
        except Exception as e:
            # Log any exceptions
            logger.exception(f"Exception raised in {self.__wrapped__.__name__}. Exception: {e!s}")
            raise


# Using the decorator
@LoggingDecorator
def divide(a, b):
    return a / b


@LoggingDecorator(log_level=logging.DEBUG)
def multiply(a, b):
    return a * b


# Call the decorated functions
result1 = divide(10, 2)
print(f"Result 1: {result1}")

result2 = multiply(5, 3)
print(f"Result 2: {result2}")

# Demonstrate exception handling
try:
    divide(10, 0)
except ZeroDivisionError:
    print("Caught division by zero error")

2025-09-18 11:57:28,842 - function_logger - INFO - Calling divide(10, 2)
2025-09-18 11:57:28,843 - function_logger - INFO - divide returned 5.0
2025-09-18 11:57:28,844 - function_logger - INFO - Calling divide(10, 0)
2025-09-18 11:57:28,845 - function_logger - ERROR - Exception raised in divide. Exception: division by zero
Traceback (most recent call last):
  File "C:\Users\FongA\AppData\Local\Temp\ipykernel_3348\1629659285.py", line 22, in __call__
    result = self.__wrapped__(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\FongA\AppData\Local\Temp\ipykernel_3348\1629659285.py", line 34, in divide
    return a / b
           ~~^~~
ZeroDivisionError: division by zero


Result 1: 5.0
Result 2: 15
Caught division by zero error


### Example 2: Creating a Retry Decorator

Let's create a decorator that retries a function call if it raises an exception:

In [7]:
import random


# Creating a retry decorator
class RetryDecorator(BaseDecorator):
    def __init__(self, func=None, max_retries=3, retry_delay=1, exceptions=(Exception,)) -> None:
        super().__init__(func)
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.exceptions = exceptions

    def __call__(self, *args, **kwargs):
        retries = 0
        while True:
            try:
                return self.__wrapped__(*args, **kwargs)
            except self.exceptions as e:
                retries += 1
                if retries > self.max_retries:
                    print(f"Maximum retries ({self.max_retries}) exceeded. Raising exception.")
                    raise
                print(f"Attempt {retries} failed with error: {e!s}. Retrying in {self.retry_delay} seconds...")
                time.sleep(self.retry_delay)


# A function that sometimes fails
@RetryDecorator(max_retries=5, retry_delay=0.5, exceptions=(ValueError,))
def unstable_function() -> str:
    # Simulate a function that sometimes fails
    if random.random() < 0.7:  # 70% chance of failure
        msg = "Random failure occurred"
        raise ValueError(msg)
    return "Success!"


# Call the decorated function
try:
    result = unstable_function()
    print(f"Result: {result}")
except ValueError:
    print("Function failed despite retries")

Result: Success!


### Example 3: Creating a Memoization Decorator

Let's create a decorator that caches the results of function calls to avoid redundant computations:

In [8]:
# Creating a memoization decorator
class MemoizeDecorator(BaseDecorator):
    def __init__(self, func=None) -> None:
        super().__init__(func)
        self.cache = {}

    def __call__(self, *args, **kwargs):
        # Create a cache key from the arguments
        key = str(args) + str(sorted(kwargs.items()))

        # Check if the result is already cached
        if key not in self.cache:
            print(f"Cache miss for {self.__wrapped__.__name__}{args}. Computing result...")
            self.cache[key] = self.__wrapped__(*args, **kwargs)
        else:
            print(f"Cache hit for {self.__wrapped__.__name__}{args}. Returning cached result.")

        return self.cache[key]


# A computationally expensive function
@MemoizeDecorator
def fibonacci(n):
    print(f"Computing fibonacci({n})")
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


# Call the decorated function multiple times
print(f"Result: {fibonacci(10)}")
print("\nCalling again with the same argument:")
print(f"Result: {fibonacci(10)}")

Cache miss for fibonacci(10,). Computing result...
Computing fibonacci(10)
Cache miss for fibonacci(9,). Computing result...
Computing fibonacci(9)
Cache miss for fibonacci(8,). Computing result...
Computing fibonacci(8)
Cache miss for fibonacci(7,). Computing result...
Computing fibonacci(7)
Cache miss for fibonacci(6,). Computing result...
Computing fibonacci(6)
Cache miss for fibonacci(5,). Computing result...
Computing fibonacci(5)
Cache miss for fibonacci(4,). Computing result...
Computing fibonacci(4)
Cache miss for fibonacci(3,). Computing result...
Computing fibonacci(3)
Cache miss for fibonacci(2,). Computing result...
Computing fibonacci(2)
Cache miss for fibonacci(1,). Computing result...
Computing fibonacci(1)
Cache miss for fibonacci(0,). Computing result...
Computing fibonacci(0)
Cache hit for fibonacci(1,). Returning cached result.
Cache hit for fibonacci(2,). Returning cached result.
Cache hit for fibonacci(3,). Returning cached result.
Cache hit for fibonacci(4,). Retu

## API Highlights

The `BaseDecorator` class provides the following key features:

- **Dual-mode behavior**: Can be used as a simple decorator (`@decorator`) or a parameterized decorator (`@decorator(arg1=value1)`)
- **Function wrapping**: Automatically wraps the decorated function, preserving its metadata
- **Coroutine support**: Can decorate both regular functions and coroutine functions
- **Pickling support**: Includes methods for proper pickling and unpickling

Key methods:
- `__new__(cls, *args, func=None, _return_partial=True, **kwargs)`: Creates either a decorator instance or a factory for creating decorator instances
- `create_decorator(cls, func, args, kwargs)`: A static method for creating a decorator instance
- `__reduce__()`: Enables decorator instances to be properly pickled and unpickled

For the full API documentation, refer to the baseobjects documentation.

## Troubleshooting / FAQs

### Q: Why is my decorator not working with async functions?

A: When decorating async functions, your decorator's `__call__` method must also be async. For example:

```python
class AsyncDecorator(BaseDecorator):
    async def __call__(self, *args, **kwargs):
        # Do something before
        result = await self.__wrapped__(*args, **kwargs)
        # Do something after
        return result
```

### Q: How do I access the original function from within my decorator?

A: The original function is stored in the `func` attribute of the `BaseDecorator` instance. You can access it using `self.__wrapped__`.

### Q: Can I use BaseDecorator with class methods or static methods?

A: Yes, `BaseDecorator` works with class methods and static methods. However, you need to apply the decorators in the correct order. For example:

```python
class MyClass:
    @classmethod
    @MyDecorator
    def my_class_method(cls, arg):
        # Method implementation
        pass
```

### Q: How do I preserve the metadata of the decorated function?

A: `BaseDecorator` automatically preserves the metadata of the decorated function, such as its name, docstring, and annotations. You don't need to use `functools.wraps` or similar utilities.

## Conclusion and Next Steps

In this tutorial, we've explored the `BaseDecorator` class from the baseobjects package. We've learned how to use it to create both simple decorators and decorators that accept arguments, and how to handle both regular functions and coroutine functions.

Key takeaways:
- `BaseDecorator` provides a foundation for creating Python decorators with extended functionality
- It supports both standard decorator usage and parameterized decorator usage
- It handles both regular functions and coroutine functions
- It includes methods for proper pickling and unpickling

Next steps:
- Explore other decorator-related classes in the baseobjects package
- Create your own custom decorators by subclassing `BaseDecorator`
- Combine `BaseDecorator` with other components of the baseobjects package
- Check out the examples directory for more examples of using `BaseDecorator`

For more information, refer to the baseobjects documentation and examples.